In [ ]:
#EDA(Exploratory Data Analysis) / Customer Analysis

%pip install pandas openpyxl
import pandas as pd
import openpyxl
from IPython.display import display

# Path στο cleaned dataset
file_path =r"../data/edited/online_retail_cleaned.xlsx"

# Διαβάζουμε το Excel
df = pd.read_excel(file_path, engine='openpyxl')



df.head()
# =========================================
# 🔍 1. Sanity Check & Initial Validation
# =========================================

df.info()
display(df.head())
display(df.describe())
display(df.shape)

# Percentage of missing values per column
(df.isna().mean() * 100).sort_values(ascending=False)


# =========================================
# 📦 2. Quantity & Unit Price Validation
# =========================================

display(df[['Quantity', 'UnitPrice']].describe())

# Check for invalid values
display(df[df['Quantity'] <= 0].shape)
display(df[df['UnitPrice'] <= 0].shape)

# Inspect highest prices
df.sort_values('UnitPrice', ascending=False).head(20)


# =========================================
# 💰 3. Revenue Calculation & Validation
# =========================================

# Create Revenue column
df['Revenue'] = df['Quantity'] * df['UnitPrice']

display(df.head())
display(df['Revenue'].describe())

# Check for negative revenue
df[df['Revenue'] < 0].head()

# Save updated dataset with Revenue
df.to_excel(
    r"../data/edited/online_retail_cleaned.xlsx"
)


# =========================================
# 📅 4. Date Validation
# =========================================

df['InvoiceDate'].min(), df['InvoiceDate'].max()
df['InvoiceDate'].isna().sum()

# Inspect earliest records
df.sort_values('InvoiceDate').head()


# =========================================
# 👤 5. CustomerID Validation
# =========================================

df['CustomerID'].isna().sum()

# Inspect missing CustomerIDs (if any)
df[df['CustomerID'].isna()][['InvoiceNo', 'Country']].head()


# =========================================
# 🌍 6. Country Distribution
# =========================================

display(df['Country'].value_counts())


# =========================================
# ⚠️ 7. Extreme Value Analysis
# =========================================

# Quantile analysis
display(df[['Quantity', 'UnitPrice', 'Revenue']].quantile([0.95, 0.99, 0.999]))

# Top revenue transactions
display(df.sort_values('Revenue', ascending=False).head(10)[
    ['InvoiceNo', 'CustomerID', 'Country', 'Quantity', 'UnitPrice', 'Revenue']
])


# =========================================
# 📦 8. Order-Level Analysis
# =========================================

# Highest quantity orders
display(df.sort_values('Quantity', ascending=False).head(10)[
    ['InvoiceNo', 'Quantity', 'UnitPrice', 'Revenue']
])

# Highest price items
df.sort_values('UnitPrice', ascending=False).head(10)[
    ['InvoiceNo', 'Quantity', 'UnitPrice', 'Revenue']
]


# =========================================
# 💸 9. Revenue Distribution Analysis
# =========================================

top_rev = (
    df.groupby('CustomerID')['Revenue']
      .sum()
      .sort_values(ascending=False)
)

display(top_rev.head(10))

# Top revenue transactions
display(df.sort_values('Revenue', ascending=False).head(10)[
    ['InvoiceNo', 'CustomerID', 'Country', 'Quantity', 'UnitPrice', 'Revenue']
])

# % contribution of top 10 customers
display(top_rev.head(10).sum() / top_rev.sum())


# =========================================
# ⏱️ 10. Time-Based Feature Engineering
# =========================================

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])

df['Year'] = df['InvoiceDate'].dt.year
df['Month'] = df['InvoiceDate'].dt.month
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')
df['DayOfWeek'] = df['InvoiceDate'].dt.dayofweek

df.head()


# =========================================
# 📊 11. Monthly KPIs
# =========================================

# Monthly Revenue
monthly_revenue = (
    df.groupby('YearMonth')['Revenue']
      .sum()
      .reset_index()
)

# Monthly Orders
monthly_orders = (
    df.groupby('YearMonth')['InvoiceNo']
      .nunique()
      .reset_index(name='Orders')
)

# Merge revenue & orders
monthly_aov = (
    monthly_revenue
        .merge(monthly_orders, on='YearMonth')
)

# Average Order Value
monthly_aov['AOV'] = monthly_aov['Revenue'] / monthly_aov['Orders']


# =========================================
# 🛒 12. Basket Size Analysis
# =========================================

items_per_order = (
    df.groupby('InvoiceNo')['Quantity']
      .sum()
      .reset_index(name='Items')
)

display(items_per_order.head())
display(items_per_order.describe())

items_per_order['Items'].quantile([0.5, 0.75, 0.9, 0.99])


# =========================================
# 📈 13. Final Monthly KPI Table
# =========================================

monthly_kpis = (
    monthly_revenue
    .merge(monthly_orders, on='YearMonth')
)

monthly_items = (
    df.groupby('YearMonth')['Quantity']
      .sum()
      .reset_index(name='Items')
)

monthly_kpis = (
    monthly_kpis
    .merge(monthly_items, on='YearMonth')
)

monthly_kpis['AOV'] = monthly_kpis['Revenue'] / monthly_kpis['Orders']

# Export KPIs
monthly_kpis.to_excel(
    r"../data/exports/monthly_kpis.xlsx",
    index=False
)

display(monthly_kpis)

df.head()